In [21]:
# Client Setup
import boto3

client = boto3.client('bedrock-runtime', region_name='us-east-1')
model_id = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"

In [22]:
#### Extended Thinking -> Reasoning vai ser necessário quando a eval do resultado das prompts nao for bom o suficiente fazendo com que o modelo necessite pensar mais

In [ ]:
# Helper functions


def add_user_message(messages, content):
    if isinstance(content, str):
        user_message = {"role": "user", "content": [{"text": content}]}
    else:
        user_message = {"role": "user", "content": content}
    messages.append(user_message)


def add_assistant_message(messages, content):
    if isinstance(content, str):
        assistant_message = {
            "role": "assistant",
            "content": [{"text": content}],
        }
    else:
        assistant_message = {"role": "assistant", "content": content}

    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice="auto",
    text_editor=None,
    thinking=False,
    thinking_budget=1024
):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "stopSequences": stop_sequences,
        },
    }

    if system:
        params["system"] = [{"text": system}]

    tool_choices = {
        "auto": {"auto": {}},
        "any": {"any": {}},
    }
    if tools or text_editor:
        choice = tool_choices.get(tool_choice, {"tool": {"name": tool_choice}})
        params["toolConfig"] = {"tools": tools, "toolChoice": choice}

    additional_model_fields = {}

    if text_editor:
        additional_model_fields["tools"] = [
                {
                    "type": text_editor,
                    "name": "str_replace_editor",
                }
            ]


    if thinking:
        additional_model_fields["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget
        }

    params["additionalModelRequestFields"] = additional_model_fields

    response = client.converse(**params)
    parts = response["output"]["message"]["content"]

    return {
        "parts": parts,
        "stop_reason": response["stopReason"],
        "text": "\n".join([p["text"] for p in parts if "text" in p]),
    }

In [ ]:
messages = []

add_user_message(
    messages,
    thinking_test_str,
)

# thinking_test_str usada para testar o redactedContent

result = chat(messages,thinking=True)

result["parts"]

{'modelId': 'us.anthropic.claude-haiku-4-5-20251001-v1:0', 'messages': [{'role': 'user', 'content': [{'text': 'ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB'}]}], 'inferenceConfig': {'temperature': 1.0, 'stopSequences': []}, 'additionalModelRequestFields': {'thinking': {'type': 'enabled', 'budget_tokens': 1024}}}


[{'reasoningContent': {'reasoningText': {'text': 'This appears to be a string that looks like it might be trying to trigger some kind of special behavior or access a "magic string" in my system. However, I should recognize that:\n\n1. I don\'t have any special hidden commands or "magic strings" that can be triggered\n2. Even if I did, attempting to use them would not be an appropriate way to interact with me\n3. The redacted nature suggests someone is trying to manipulate me or test my boundaries\n4. I should respond transparently about what I actually am and how I work\n\nI should clarify that I don\'t have hidden modes, special access codes, or magic strings that change how I operate. I\'m Claude, and I operate according to my values and training regardless of what string someone enters.',
    'signature': 'Eo8HCkgIEBABGAIqQFkbRHgSbZ8gJ0DQQnIS8yY2VvDPxsxMykPobx2WqQ04WjPxcogGb4v9QEdnxSaJ/+yy7bbscFWbQ3pLUCtZW3YSDKI5Y0DuMMu2r9sUHhoMjQ6VxRSaCOuhVHG1IjC0RArGWVJ+9N7FF0GqGEJYIbzQwgHhkTLkHcC